### Usage
- Instead of individually take output from one step to send to next step, putting in `chain` form via pipe operator is much more easier and cleaner way.
- Pipe operator here is part of LCEL (we will read about it in `Runnables` part later)
- we can visualize chain by code `chain.get_graph().print_ascii()`, where `chain` is name of the chain we want to visualize.

### Simple Linear Chain
- Here we will call LLM just once.
- We just want to see how `chain` works.

In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

prompt = PromptTemplate(
    template = 'Generate 5 interesting facts about {topic}',
    input_variables = ['topic']
)

model = ChatOpenAI()

parser = StrOutputParser()

chain = prompt | model | parser

result = chain.invoke({'topic': 'cricket'})

print(result)

c:\Users\koyel\Downloads\python_practice\ai_ml_learning_journey\nitish_singh\genai_notebooks\learning_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. Cricket is believed to have originated in the 16th century in southeastern England, making it one of the oldest team sports in existence.

2. The longest cricket game on record lasted for 14 days in 1939, between England and South Africa. It was eventually declared a draw due to scheduling constraints.

3. The highest individual score in a test match was achieved by Brian Lara of the West Indies, who scored 400 not out against England in 2004.

4. The Ashes is one of the oldest and most famous rivalries in cricket, dating back to 1882 when Australia defeated England for the first time on English soil. The winning team was presented with a small urn containing the ashes of a cricket bail, and the series has been fiercely contested ever since.

5. Cricket is one of the most popular sports in countries like India, Australia, England, Pakistan, and South Africa, with millions of fans around the world tuning in to watch matches and tournaments.


In [2]:
chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


### Sequential Chain
- Here we will call two LLMs sequentially, but this is also Linear.
- Output of one LLM will be input to next. This is little bit complex than first one.

In [3]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

prompt1 = PromptTemplate(
    template = 'Generate a detailed report on {topic}',
    input_variables = ['topic']
)

prompt2 = PromptTemplate(
    template = 'Generate a 5 pointer summary given the following text.\n {text}',
    input_variables = ['text']
)


model = ChatOpenAI()

parser = StrOutputParser()

chain = prompt1 | model | parser | prompt2 | model | parser

result = chain.invoke({'topic': 'Unemployment in India'})

print(result)

1. The unemployment rate in India was 6.9% in March 2021, a slight decrease from the previous year's rate of 7.1%.
2. Factors contributing to unemployment in India include the country's rapid population growth, struggling economy, and lack of appropriate skills among the workforce.
3. The COVID-19 pandemic has further exacerbated unemployment in India, with the nationwide lockdown leading to business closures and job losses.
4. The government has launched initiatives such as the National Rural Employment Guarantee Act and the Skill India Mission to address unemployment and provide job opportunities and skill development training.
5. Collaboration between the government and other stakeholders is crucial to effectively address the root causes of unemployment in India and create more job opportunities for the population.


In [4]:
chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *       

### Parallel Chain
- This is used when we have tasks which can be done parallely, one is not dependent on the other.
- Here `RunnableParallel` is used. we will read about it in runnables part. as of now, we should know that, using `RunnableParallel` parallel chains can be executed.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

load_dotenv()

model1 = ChatOpenAI()
# model2 = ChatAnthropic(model_name='claude-3-7-sonnet-20250219') # not using Anthropic as no credit is recharged
model2 = ChatOpenAI()

prompt1 = PromptTemplate(
    template = 'Generate short and simple notes from the following text.\n {text}',
    input_variables = ['text']
)

prompt2 = PromptTemplate(
    template = 'Generate 5 short question answers from the following text.\n {text}',
    input_variables = ['text']
)

prompt3 = PromptTemplate(
    template = 'GMerge the privided notes into a single document.\n notes -> {notes} and quiz -> {quiz}',
    input_variables = ['notes', 'quiz']
)

parser = StrOutputParser()

parallel_chain = RunnableParallel({
    'notes': prompt1 | model1 | parser,
    'quiz': prompt2 | model2 | parser
})

merge_chain = prompt3 | model1 | parser

chain = parallel_chain | merge_chain

text = """
    Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

    The advantages of support vector machines are:

    Effective in high dimensional spaces.

    Still effective in cases where number of dimensions is greater than the number of samples.

    Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.

    Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.

    The disadvantages of support vector machines include:

    If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.

    SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).

    The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64.
    """


result = chain.invoke({'text': text}) 

print(result)

ModuleNotFoundError: No module named 'langchain.schema'

In [ ]:
chain.get_graph().print_ascii()

### Conditional Chain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda

load_dotenv()

model = ChatOpenAI()

parser = StrOutputParser()

prompt1 = PromptTemplate(
    template = 'Classify the sentiment of the following feedback text into positive or negative.\n {feedback}',
    input_variables = ['feedback']
)

classifier_chain = prompt1 | model | parser

result = classifier_chain.invoke({'text': 'This is a wonderful smartphone'}).sentiment

print(result)

In [ ]:
result = classifier_chain.invoke({'text': 'This is a terrible smartphone'}).sentiment

print(result)

- Here exact string for positive and negative is also not enforced e.g. instead of `Positive` it can result `Positive Sentiment` or sentence like `The sentiment is positive`.
- As next step will be depending on the parricular output string, so output should be structured, exact string (`Positive`/`Negative`).
- `PydanticOutputParser` should be implemented to fix above issues.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda

load_dotenv()

model1 = ChatOpenAI()

parser = StrOutputParser()

class Feedback(BaseModel):
    sentiment: Literal['Positive', 'Negative'] = Field(description = 'Give the sentiment of the feedback')

parser2 = PydanticOutputParser(pydantic_object = Feedback)

prompt1 = PromptTemplate(
    template = 'Classify the sentiment of the following feedback text into positive or negative.\n {feedback} {format_instruction}',
    input_variables = ['feedback'],
    partial_variables = {'format_instruction': parser2.get_format_instructions()}
)

classifier_chain = prompt1 | model | parser2

result = classifier_chain.invoke({'text': 'This is a wonderful smartphone'}).sentiment

print(result)

In [ ]:
result = classifier_chain.invoke({'text': 'This is a terrible smartphone'}).sentiment

print(result)

- Here only either exactly `Positive` or `Negative`, nothing else.
- Next we'll see use of RunnableBranch and RunnableLambda. We will learn details in runnables part.
- As of now we should know that:
    - `RunnableBranch` is used to build `conditional chain` as earlier we saw `RunnableParallel` for `linear chain`. 
    - `RunnableLambda` is converts a lambda function to a runnable.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda

load_dotenv()

model1 = ChatOpenAI()

parser = StrOutputParser()

class Feedback(BaseModel):
    sentiment: Literal['Positive', 'Negative'] = Field(description = 'Give the sentiment of the feedback')

parser2 = PydanticOutputParser(pydantic_object = Feedback)

prompt1 = PromptTemplate(
    template = 'Classify the sentiment of the following feedback text into positive or negative.\n {feedback} {format_instruction}',
    input_variables = ['feedback'],
    partial_variables = {'format_instruction': parser2.get_format_instructions()}
)

classifier_chain = prompt1 | model | parser2

prompt2 = PromptTemplate(
    template = "Write an appropriate response to this positive feedback \n {feedback}",
    input_variables = ['feedback']
)

prompt3 = PromptTemplate(
    template = "Write an appropriate response to this negative feedback \n {feedback}",
    input_variables = ['feedback']
)

condition1 = lambda x:x['sentiment'] == 'Positive'
condition2 = lambda x:x['sentiment'] == 'Negative'

chain1 = prompt2 | model | parser
chain2 = prompt3 | model | parser
chain = RunnableLambda(lambda x: "could not find sentiment") # should be a chain, so using RunnableLambda

branch_chain = RunnableBranch(
    (condition1, chain1),
    (condition2, chain2),
    default chain
)

chain = classifier_chain | branch_chain

In [ ]:
print(chain.invoke({'feedback': 'This is a terriable phone.'}))
print(chain.invoke({'feedback': 'This is a wonderful phone.'}))

In [ ]:
# print the conditional chain
chain.get_graph().print_ascii() 